<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M04_01_Correlacion_causalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos

## Módulo 4: Análisis e interpretación de datos

### Notebook 1: Correlación y causalidad

En los módulos anteriores aprendimos a importar, organizar, limpiar y explorar datos. También utilizamos herramientas de estadística descriptiva y visualización para comprender el comportamiento de variables ambientales.

Hasta ahora, nuestros análisis se han centrado principalmente en estudiar las características de una variable individual.

Sin embargo, en ingeniería agrícola y ambiental frecuentemente necesitamos comprender **cómo se relacionan diferentes variables**.

Por ejemplo:

- ¿Existe una relación entre la temperatura del aire y la humedad relativa?
- ¿Cómo cambia la temperatura cuando aumenta la radiación solar?
- ¿La presión atmosférica está relacionada con otras variables meteorológicas?
- ¿Una relación estadística significa necesariamente que una variable causa cambios en otra?

Para responder estas preguntas utilizaremos datos reales de la estación meteorológica de bajo costo **MeteoChillán**, ubicada en el Campus Chillán de la Universidad de Concepción.

En este notebook aprenderemos a explorar y cuantificar relaciones entre variables, además de reconocer las limitaciones de su interpretación.

Al finalizar este notebook podrás:

- Identificar relaciones entre variables mediante diagramas de dispersión.
- Distinguir relaciones positivas, negativas y no lineales.
- Calcular e interpretar coeficientes de correlación de Pearson y Spearman.
- Construir e interpretar matrices de correlación.
- Evaluar cómo los valores extremos pueden influir en una correlación.
- Comprender por qué **correlación no implica causalidad**.

## 1. Importar las librerías

Utilizaremos las bibliotecas que ya conocemos:

- `pandas`: para organizar y analizar los datos.
- `matplotlib.pyplot`: para configurar y personalizar figuras.
- `seaborn`: para construir visualizaciones estadísticas.

Mantendremos un formato gráfico consistente para facilitar la comparación e interpretación de los resultados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr, spearmanr

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

## 2. Cargar los datos de MeteoChillán

Utilizaremos los datos de la estación meteorológica de bajo costo **MeteoChillán**, ubicada en el Campus Chillán de la Universidad de Concepción.

Esta estación registra diferentes variables meteorológicas, entre ellas:

- Temperatura del aire (°C).
- Humedad relativa (%).
- Presión atmosférica (hPa).
- Velocidad del viento (m/s).
- Radiación solar (W/m²).
- Precipitación (mm).

En este notebook analizaremos cómo se relacionan algunas de estas variables.

Para ello, necesitamos que las observaciones correspondan a los **mismos instantes de tiempo**. No tendría sentido comparar la temperatura de las 10:00 con la humedad relativa de las 15:00 como si fueran observaciones simultáneas.

Utilizaremos el archivo de MeteoChillán empleado anteriormente en el Módulo 2.

**Fuente:** [MeteoChillán — Universidad de Concepción](https://meteochillan.udec.cl/stations.html)

In [ ]:
from google.colab import files

uploaded = files.upload()

nombre_archivo = next(iter(uploaded))
datos = pd.read_csv(nombre_archivo)

print("Dimensiones:", datos.shape)
print("\nColumnas disponibles:")
print(datos.columns.tolist())

datos.head()

### Selección de variables

Para analizar las correlaciones, seleccionaremos seis variables meteorológicas:

- Temperatura del aire (°C).
- Humedad relativa (%).
- Radiación solar (W/m²).
- Velocidad del viento (m/s).
- Presión atmosférica (hPa).
- Precipitación horaria (mm).

Organizaremos estas variables en un nuevo DataFrame, utilizando la fecha como índice temporal.

También revisaremos si existen datos faltantes antes de comenzar el análisis.

### Actividad práctica: preparación y control de calidad de los datos

Antes de analizar las correlaciones, debemos preparar los datos de MeteoChillán.

El archivo contiene 27 columnas, pero no todas son necesarias para nuestro análisis. Además, debemos comprobar que las observaciones sean válidas y que las variables puedan compararse temporalmente.

**Trabaja con las siguientes variables:**

- Temperatura del aire (°C).
- Humedad relativa (%).
- Radiación solar (W/m²).
- Velocidad del viento (m/s).
- Presión atmosférica (hPa).
- Precipitación (mm).

**Actividades**

1. **Selección de variables**
   - Identifica qué columnas del archivo corresponden a las seis variables indicadas.
   - Construye un nuevo DataFrame con las variables seleccionadas.

2. **Preparación temporal**
   - Convierte la columna de fecha al formato `datetime`.
   - Establece la fecha como índice del DataFrame.
   - Ordena los registros cronológicamente.
   - Determina la fecha inicial y final de la serie.
   - Identifica la frecuencia temporal de los registros.

3. **Control de calidad**
   - Identifica los valores faltantes (`NaN`) de cada variable.
   - Comprueba si existen fechas duplicadas.
   - Identifica las fechas esperadas que no están presentes en la serie.
   - Revisa los valores mínimos y máximos de cada variable.
   - Identifica posibles valores físicamente inconsistentes.

4. **Preparación para el análisis**
   - Decide cómo tratar los valores faltantes y los registros inconsistentes.
   - Comprueba que las variables seleccionadas correspondan a los mismos instantes de tiempo.
   - Prepara un DataFrame final para el análisis de correlación.


**Resultado esperado:** un DataFrame con las variables seleccionadas, fechas correctamente organizadas y un control de calidad básico documentado.

### 2.1. Exploración inicial de los datos

Antes de calcular correlaciones, debemos conocer las características de nuestra base de datos.

Aunque las observaciones provienen de una misma estación meteorológica, no debemos asumir que todos los registros tienen una frecuencia temporal constante ni que todas las variables representan el mismo tipo de medición.

Primero revisaremos:

- Las fechas inicial y final del registro.
- Los tipos de datos.
- Los intervalos de tiempo entre observaciones.
- La existencia de fechas duplicadas.
- La presencia de datos faltantes.

Posteriormente, realizaremos un control de calidad básico y construiremos una base de datos con resolución horaria.

In [ ]:
# Convertir la columna de fecha
datos['datetime_utc'] = pd.to_datetime(
    datos['datetime_utc'],
    utc=True,
    errors='coerce'
)

# Revisar dimensiones y tipos de datos
print("Dimensiones:", datos.shape)

print("\nTipos de datos:")
print(datos.dtypes)

# Revisar periodo disponible
print("\nFecha inicial:", datos['datetime_utc'].min())
print("Fecha final:", datos['datetime_utc'].max())

# Revisar fechas inválidas
print("\nFechas inválidas:", datos['datetime_utc'].isna().sum())

# Revisar duplicados
print("Fechas duplicadas:", datos['datetime_utc'].duplicated().sum())

### 2.2. Frecuencia temporal de las observaciones

La frecuencia temporal indica cada cuánto tiempo se registra una observación.

Por ejemplo:

- Cada 5 minutos: 12 observaciones por hora.
- Cada 15 minutos: 4 observaciones por hora.
- Cada 30 minutos: 2 observaciones por hora.
- Cada 60 minutos: 1 observación por hora.

Una serie puede contener registros con diferentes intervalos debido a cambios en la configuración de la estación, interrupciones o problemas en la adquisición de datos.

Para identificar la frecuencia real, calcularemos las diferencias de tiempo entre observaciones consecutivas.

In [ ]:
# Ordenar cronológicamente
datos = datos.sort_values('datetime_utc')

# Calcular diferencias entre registros consecutivos
diferencias = datos['datetime_utc'].diff()

# Mostrar frecuencias más comunes
print("Intervalos entre observaciones:")

display(
    diferencias.value_counts().head(15)
)

# Convertir diferencias a minutos
intervalos_min = diferencias.dt.total_seconds() / 60

print("\nIntervalos en minutos:")

display(
    intervalos_min.value_counts().head(15)
)

### 2.3. Identificación de fechas faltantes

Los valores faltantes pueden presentarse de dos formas:

1. Existe una fecha en el archivo, pero una variable tiene un valor `NaN`.
2. La fecha completa no aparece en la base de datos.

El segundo caso no puede identificarse únicamente utilizando `isna()`.

Para identificar fechas ausentes, construiremos una secuencia temporal regular y la compararemos con las fechas disponibles.

Utilizaremos inicialmente una frecuencia de 30 minutos como referencia para esta revisión.

In [ ]:
# Crear índice temporal
fechas = pd.DatetimeIndex(
    datos['datetime_utc'].dropna()
)

# Contar observaciones originales por hora
observaciones_hora = (
    pd.Series(1, index=fechas)
    .resample('1h')
    .sum()
    .fillna(0)
    .astype(int)
)

print("Distribución de observaciones por hora:")
display(observaciones_hora.value_counts().sort_index())

print("\nHoras esperadas:", len(observaciones_hora))
print("Horas sin observaciones:", (observaciones_hora == 0).sum())

# Identificar horas sin observaciones
horas_faltantes = observaciones_hora[
    observaciones_hora == 0
]

display(horas_faltantes.head(20))

### 2.4. Selección de variables meteorológicas

El archivo contiene 27 columnas. Algunas representan variables meteorológicas diferentes, mientras que otras corresponden a distintas formas de expresar o acumular una misma variable.

Para nuestro análisis seleccionaremos:

- Temperatura del aire (°C).
- Humedad relativa (%).
- Radiación solar (W/m²).
- Velocidad del viento (m/s).
- Presión atmosférica absoluta (hPa).

La precipitación requiere especial atención porque el archivo contiene distintas variables relacionadas con ella:

- `rain_rate_mm_hr`: intensidad de precipitación (mm/h).
- `hourly_rain_mm`: acumulación de precipitación horaria (mm).
- `daily_rain_mm`: acumulación diaria (mm).
- `event_rain_mm`: acumulación durante un evento (mm).

No debemos sumar directamente estas columnas sin conocer cómo se calculan y actualizan sus valores.

Por ahora trabajaremos con las cinco variables meteorológicas continuas. La precipitación podrá incorporarse posteriormente, una vez definido un procedimiento adecuado para obtener acumulaciones por intervalo.

In [ ]:
# Seleccionar variables
columnas = [
    'datetime_utc',
    'air_temperature_c',
    'relative_humidity_pct',
    'solar_radiation_w_m2',
    'wind_speed_ms',
    'absolute_pressure_hpa'
]

df = datos[columnas].copy()

# Establecer índice temporal
df = df.set_index('datetime_utc').sort_index()

# Renombrar variables
df = df.rename(columns={
    'air_temperature_c': 'Temperatura',
    'relative_humidity_pct': 'Humedad',
    'solar_radiation_w_m2': 'Radiación',
    'wind_speed_ms': 'Viento',
    'absolute_pressure_hpa': 'Presión'
})

display(df.head())

### 2.5. Exploración visual de las series meteorológicas

Antes de realizar el control de calidad, es importante visualizar cómo evolucionan las variables meteorológicas a lo largo del tiempo.

Los gráficos de series temporales permiten identificar características que no siempre son evidentes al revisar tablas o estadísticos descriptivos.

En particular, observaremos:

- Variaciones temporales y posibles patrones estacionales.
- Valores extremos y cambios bruscos.
- Periodos sin observaciones.
- Valores que podrían indicar errores de medición.
- Diferencias en el comportamiento temporal de las variables.

Representaremos las cinco variables meteorológicas en gráficos independientes, utilizando el periodo completo disponible.

**Preguntas para discutir:**

1. ¿Qué variables presentan mayor variabilidad temporal?
2. ¿Se observan diferencias entre verano e invierno?
3. ¿Existen valores extremos que deberían revisarse?
4. ¿Se identifican periodos sin observaciones?
5. ¿Qué variables parecen presentar patrones temporales similares?

La exploración visual permite identificar posibles problemas, pero no constituye por sí sola una validación de la calidad de los datos.

In [ ]:


# Variables y unidades
variables = {
    'Temperatura': 'Temperatura (°C)',
    'Humedad': 'Humedad relativa (%)',
    'Radiación': 'Radiación solar (W/m²)',
    'Viento': 'Velocidad del viento (m/s)',
    'Presión': 'Presión atmosférica (hPa)'
}

# Crear gráficos
fig, axes = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=(14, 14),
    sharex=True
)

for ax, (variable, unidad) in zip(axes, variables.items()):

    ax.plot(
        df.index,
        df[variable],
        linewidth=0.6
    )

    ax.set_ylabel(unidad, fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='both', labelsize=10)

# Formato de fechas
axes[-1].xaxis.set_major_locator(
    mdates.MonthLocator(interval=1)
)

axes[-1].xaxis.set_major_formatter(
    mdates.DateFormatter('%b %Y', tz=mdates.UTC)
)

axes[-1].set_xlabel('Fecha (UTC)', fontsize=12)

fig.suptitle(
    'Series meteorológicas — MeteoChillán, Campus Chillán UdeC',
    fontsize=15
)

plt.tight_layout()
plt.show()

### 2.6. Control de calidad

Antes de analizar las relaciones entre variables, revisaremos si existen datos faltantes o valores físicamente inconsistentes.

Aplicaremos comprobaciones básicas:

- Humedad relativa entre 0 y 100 %.
- Radiación solar no negativa.
- Velocidad del viento no negativa.
- Presión atmosférica positiva.
- Temperatura dentro de un intervalo amplio de valores plausibles.

Estos controles permiten identificar errores evidentes, pero no garantizan que todas las observaciones sean correctas.

In [ ]:
# Convertir variables a formato numérico
df = df.apply(pd.to_numeric, errors='coerce')

# Revisar valores faltantes
print("Valores faltantes:")
display(df.isna().sum())

# Estadísticos descriptivos
print("\nEstadísticos descriptivos:")
display(df.describe().T)

# Identificar valores fuera de rangos básicos
control = pd.DataFrame({
    'Temperatura fuera de rango': [
        ((df['Temperatura'] < -20) |
         (df['Temperatura'] > 50)).sum()
    ],
    'Humedad fuera de rango': [
        ((df['Humedad'] < 0) |
         (df['Humedad'] > 100)).sum()
    ],
    'Radiación negativa': [
        (df['Radiación'] < 0).sum()
    ],
    'Viento negativo': [
        (df['Viento'] < 0).sum()
    ],
    'Presión no positiva': [
        (df['Presión'] <= 0).sum()
    ]
})

display(control)

In [ ]:
# Crear copia para limpieza
df_limpio = df.copy()

# Reemplazar valores inconsistentes por NaN
df_limpio.loc[
    ~df_limpio['Temperatura'].between(-20, 50),
    'Temperatura'
] = np.nan

df_limpio.loc[
    ~df_limpio['Humedad'].between(0, 100),
    'Humedad'
] = np.nan

df_limpio.loc[
    df_limpio['Radiación'] < 0,
    'Radiación'
] = np.nan

df_limpio.loc[
    df_limpio['Viento'] < 0,
    'Viento'
] = np.nan

df_limpio.loc[
    df_limpio['Presión'] <= 0,
    'Presión'
] = np.nan

print("Valores faltantes después del control:")
display(df_limpio.isna().sum())

### 2.7. Agregación temporal a resolución horaria

Las observaciones originales pueden presentar diferentes intervalos temporales.

Para analizar las relaciones entre variables, construiremos una base de datos con una resolución común de una hora.

Utilizaremos el promedio de las observaciones disponibles dentro de cada hora para:

- Temperatura del aire.
- Humedad relativa.
- Radiación solar.
- Velocidad del viento.
- Presión atmosférica.

Por ejemplo, si existen observaciones a las 10:00 y 10:30, ambas se utilizarán para calcular el promedio correspondiente a la hora de las 10:00.

La agregación temporal también permite reducir la cantidad de registros y facilita la interpretación de las relaciones meteorológicas.

Sin embargo, una media horaria calculada con una sola observación puede ser menos representativa que otra calculada con varias observaciones. Por ello, revisaremos cuántos registros contribuyen a cada promedio.

In [ ]:
# Agregar variables a resolución horaria
df_horario = df_limpio.resample('1h').mean()

# Contar observaciones por hora
conteo_horario = df_limpio.resample('1h').count()

print("Dimensiones originales:", df_limpio.shape)
print("Dimensiones horarias:", df_horario.shape)

print("\nPrimeras observaciones horarias:")
display(df_horario.head())

print("\nNúmero de observaciones utilizadas por hora:")
display(conteo_horario.head())

In [ ]:
# Contar horas con al menos una observación por variable
horas_disponibles = df_horario.notna().sum()

# Contar horas sin observaciones por variable
horas_faltantes = df_horario.isna().sum()

resumen = pd.DataFrame({
    'Horas disponibles': horas_disponibles,
    'Horas faltantes': horas_faltantes,
    'Disponibilidad (%)': (
        100 * horas_disponibles / len(df_horario)
    ).round(2)
})

display(resumen)

### 2.8. Base de datos final para el análisis de correlaciones

Después del control de calidad y la agregación temporal, disponemos de una serie horaria con cinco variables meteorológicas.

La base contiene todas las horas comprendidas entre la primera y la última observación. Cuando no existen registros para una hora determinada, sus valores aparecen como `NaN`.

**No eliminaremos estas horas**, ya que queremos conservar la continuidad temporal y la información sobre los periodos sin observaciones.

Al calcular las correlaciones, utilizaremos únicamente los pares de observaciones válidas para las variables que estemos comparando.

Por ejemplo, para analizar la relación entre temperatura y humedad relativa, se utilizarán las horas en las que ambas variables tengan datos, independientemente de la disponibilidad de las demás variables.

Finalmente, guardaremos la base de datos horaria para utilizarla en los siguientes análisis.

In [ ]:
# Mantener todas las horas, incluyendo aquellas sin observaciones
df_final = df_horario.copy()

# Resumen de disponibilidad
print("Horas totales:", len(df_final))

print("\nValores faltantes por variable:")
display(df_final.isna().sum())

print("\nDisponibilidad por variable (%):")
display((100 * df_final.notna().mean()).round(2))

# Verificar estructura
print("\nPrimeras observaciones:")
display(df_final.head())

print("\nEstadísticos descriptivos:")
display(df_final.describe().T)

In [ ]:
# Exportar base de datos final
df_final.to_csv(
    'MeteoChillan_datos_horarios.csv',
    index=True
)

print("Archivo guardado correctamente.")

### 2.9. Exploración visual de las series horarias

Una vez agregados los datos a resolución horaria, podemos visualizar la evolución temporal de las variables meteorológicas.

La serie horaria incluye todas las horas comprendidas entre la primera y la última observación. Cuando una hora no tiene datos, aparece como `NaN`, lo que permite identificar visualmente las interrupciones en los registros.

Los gráficos nos ayudarán a identificar:

- Patrones diarios y estacionales.
- Cambios bruscos y valores extremos.
- Periodos sin observaciones.
- Posibles comportamientos anómalos.

Primero analizaremos el periodo completo y posteriormente una semana para observar con mayor detalle los ciclos diarios.

In [ ]:


# Variables y unidades
variables = {
    'Temperatura': 'Temperatura (°C)',
    'Humedad': 'Humedad relativa (%)',
    'Radiación': 'Radiación solar (W/m²)',
    'Viento': 'Velocidad del viento (m/s)',
    'Presión': 'Presión atmosférica (hPa)'
}

# Crear gráficos
fig, axes = plt.subplots(
    5, 1,
    figsize=(14, 13),
    sharex=True
)

for ax, (variable, unidad) in zip(axes, variables.items()):

    ax.plot(
        df_horario.index,
        df_horario[variable],
        color='steelblue',
        linewidth=0.7
    )

    ax.set_ylabel(unidad, fontsize=11)
    ax.grid(True, alpha=0.3)

# Formato del eje temporal
axes[-1].xaxis.set_major_locator(
    mdates.MonthLocator(interval=1, tz=mdates.UTC)
)

axes[-1].xaxis.set_major_formatter(
    mdates.DateFormatter('%b %Y', tz=mdates.UTC)
)

axes[-1].set_xlabel('Fecha (UTC)', fontsize=11)

fig.suptitle(
    'Series temporales horarias — MeteoChillán',
    fontsize=14
)

plt.tight_layout()
plt.show()

## 3. Análisis de correlación entre variables meteorológicas

En las secciones anteriores preparamos y exploramos los datos meteorológicos de MeteoChillán. Ahora estudiaremos las relaciones que pueden existir entre las variables.

La **correlación** es una medida estadística que permite describir cómo varían dos variables conjuntamente.

Por ejemplo, podemos preguntarnos:

- ¿Qué ocurre con la humedad relativa cuando aumenta la temperatura?
- ¿Las horas con mayor radiación solar presentan temperaturas más altas?
- ¿Existe alguna relación entre la presión atmosférica y la velocidad del viento?

Para estudiar estas relaciones utilizaremos diagramas de dispersión y el coeficiente de correlación de Pearson.

Es importante distinguir entre **correlación y causalidad**: encontrar una relación estadística entre dos variables no demuestra que una sea la causa de los cambios en la otra.

### 3.1. ¿Cómo pueden relacionarse dos variables?

Cuando comparamos dos variables, podemos encontrar diferentes patrones:

**Correlación positiva:** cuando una variable aumenta, la otra tiende a aumentar.

**Correlación negativa:** cuando una variable aumenta, la otra tiende a disminuir.

**Correlación cercana a cero:** no se observa una relación lineal clara entre las variables.

Estas relaciones pueden identificarse visualmente mediante un **diagrama de dispersión**, donde cada punto representa un par de observaciones.

La correlación describe una tendencia general. No significa que todos los puntos deban seguir exactamente el mismo patrón.

### 3.2. Diagramas de dispersión

Un diagrama de dispersión (*scatter plot*) permite visualizar la relación entre dos variables numéricas.

Cada punto representa dos mediciones correspondientes al mismo instante de tiempo.

Comenzaremos comparando la temperatura del aire y la humedad relativa.

**Antes de observar el gráfico, responde:**

1. ¿Esperas encontrar una correlación positiva o negativa?
2. ¿Crees que la relación será fuerte o débil?
3. ¿Por qué la temperatura y la humedad relativa podrían estar relacionadas?

Después compararemos nuestras hipótesis con las observaciones de MeteoChillán.

In [ ]:
# Seleccionar las dos variables
pares = df_final[['Temperatura', 'Humedad']].dropna()

# Crear diagrama de dispersión
plt.figure(figsize=(8, 5))

plt.scatter(
    pares['Temperatura'],
    pares['Humedad'],
    alpha=0.3,
    s=12,
    color='steelblue'
)

plt.xlabel('Temperatura del aire (°C)', fontsize=11)
plt.ylabel('Humedad relativa (%)', fontsize=11)

plt.title('Relación entre temperatura y humedad relativa')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Número de pares de observaciones:', len(pares))

**Analiza el diagrama de dispersión:**

1. ¿Qué tendencia general observas entre temperatura y humedad relativa?
2. ¿La relación parece positiva o negativa?
3. ¿Los puntos se agrupan cerca de una línea o presentan mucha dispersión?
4. ¿Existen temperaturas similares con valores muy diferentes de humedad relativa?
5. ¿Podemos afirmar, únicamente a partir de este gráfico, que la temperatura causa los cambios en la humedad relativa?

**Importante:** la humedad relativa depende de la cantidad de vapor de agua presente y de la temperatura del aire. Por ello, una relación entre ambas variables puede tener una explicación física, pero el diagrama de dispersión por sí solo no permite establecer causalidad.

### 3.3. Actividad práctica: exploración de relaciones entre variables

Construir una figura con diagramas de dispersión (*scatter plots*) para explorar visualmente todas las relaciones posibles entre las cinco variables meteorológicas:

- Temperatura del aire (°C).
- Humedad relativa (%).
- Radiación solar (W/m²).
- Velocidad del viento (m/s).
- Presión atmosférica (hPa).

**Instrucciones:**

1. Utiliza el DataFrame `df_final`, que contiene los datos horarios.
2. Identifica las 10 combinaciones posibles entre las cinco variables, sin repetir pares.
3. Construye una figura utilizando `plt.subplots()` con **2 filas y 5 columnas**.
4. Representa cada combinación mediante un diagrama de dispersión utilizando `scatter()`.
5. Incluye los nombres y unidades de las variables en los ejes.
6. Utiliza puntos pequeños y cierta transparencia (`alpha`) para facilitar la visualización.
7. Asegúrate de que los valores faltantes (`NaN`) no se incluyan en cada comparación.
8. Ajusta el tamaño de la figura para que todos los gráficos sean legibles.

**Preguntas de análisis:**

1. ¿Qué pares de variables presentan una relación positiva?
2. ¿Qué pares presentan una relación negativa?
3. ¿Qué pares no muestran una relación lineal evidente?
4. ¿Se observan relaciones que no parecen lineales?
5. ¿Existen agrupaciones de puntos o valores extremos que llamen la atención?
6. ¿Qué relaciones podrían tener una explicación física?



### 3.4. Coeficientes de correlación: Pearson y Spearman

Los diagramas de dispersión permiten identificar visualmente relaciones entre variables. Sin embargo, necesitamos una medida numérica para cuantificar su dirección y fuerza.

Un **coeficiente de correlación** resume la asociación entre dos variables mediante un valor comprendido entre **−1 y +1**.

- **+1:** asociación positiva perfecta.
- **0:** ausencia de asociación del tipo que mide el coeficiente.
- **−1:** asociación negativa perfecta.

Cuanto más cercano sea el valor a +1 o −1, mayor será la fuerza de la asociación correspondiente.

Los dos coeficientes que estudiaremos son:

**Pearson (r):** mide la fuerza y dirección de una relación lineal entre dos variables numéricas.

**Spearman (ρ):** mide la fuerza y dirección de una relación monótona, utilizando los rangos de las observaciones en lugar de sus valores originales.

Una relación es **monótona** cuando, en general, una variable aumenta mientras la otra aumenta o disminuye, aunque la relación no necesariamente siga una línea recta.

Ambos coeficientes describen asociaciones estadísticas, pero ninguno demuestra causalidad.

### 3.4.1. Coeficiente de correlación de Pearson

El coeficiente de Pearson (r) mide qué tan cerca se encuentran las observaciones de una relación lineal.

Se calcula mediante:

$$
r = \frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}
\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}
$$

Donde:

- $x_i$ e $y_i$: observaciones de las dos variables.
- $\bar{x}$ e $\bar{y}$: medias de las variables.
- $n$: número de pares de observaciones válidas.

**¿Cuándo utilizar Pearson?**

- Cuando queremos evaluar una relación lineal.
- Cuando trabajamos con variables numéricas.
- Cuando el diagrama de dispersión no muestra una relación claramente curvilínea.
- Cuando los resultados no están dominados por valores extremos.

**Limitaciones:**

- Es sensible a valores extremos.
- Puede presentar valores cercanos a cero aunque exista una relación no lineal importante.
- Una correlación elevada no implica causalidad.

### 3.4.2. Coeficiente de correlación de Spearman

El coeficiente de Spearman (ρ) mide la asociación monótona entre dos variables.

A diferencia de Pearson, utiliza la posición o **rango** de cada observación en lugar de su valor original.

Por ejemplo, para los valores:

| Temperatura (°C) | Rango |
|---|---|
| 10 | 1 |
| 15 | 2 |
| 20 | 3 |
| 25 | 4 |

Spearman calcula la correlación de Pearson entre los rangos de ambas variables.

Cuando no existen rangos repetidos, también puede expresarse como:

$$
\rho = 1-\frac{6\sum_{i=1}^{n}d_i^2}{n(n^2-1)}
$$

Donde:

- $d_i$: diferencia entre los rangos de cada par de observaciones.
- $n$: número de pares de observaciones.

**¿Cuándo utilizar Spearman?**

- Cuando la relación es monótona, pero no necesariamente lineal.
- Cuando existen valores extremos que pueden afectar considerablemente a Pearson.
- Cuando trabajamos con variables ordinales.
- Cuando queremos evaluar si valores mayores de una variable tienden a corresponder con valores mayores o menores de otra.

**Importante:** Spearman tampoco detecta necesariamente relaciones no monótonas, como una relación en forma de U.

### 3.5. Interpretación de los coeficientes de correlación

Los coeficientes de Pearson y Spearman varían entre −1 y +1.

El **signo** indica la dirección de la relación y el **valor absoluto** indica su fuerza.

Como **orientación general** (no es una regla!), podemos utilizar la siguiente clasificación:

| Valor absoluto del coeficiente | Interpretación |
|---|---|
| 0.00 – 0.19 | Muy débil |
| 0.20 – 0.39 | Débil |
| 0.40 – 0.59 | Moderada |
| 0.60 – 0.79 | Fuerte |
| 0.80 – 1.00 | Muy fuerte |


La interpretación depende del fenómeno estudiado.



### 3.6. ¿La correlación es estadísticamente significativa?

Hasta ahora hemos estudiado la dirección y fuerza de las correlaciones.

Sin embargo, cuando utilizamos una muestra de observaciones, también podemos preguntarnos si la asociación encontrada proporciona evidencia de una relación en la población o si podría explicarse por la variabilidad del muestreo.

Para ello utilizamos una **prueba de hipótesis**.

#### Hipótesis estadísticas

**Hipótesis nula ($H_0$):**

No existe correlación en la población.

$$
H_0: \rho = 0
$$

**Hipótesis alternativa ($H_1$):**

Existe una correlación diferente de cero.

$$
H_1: \rho \neq 0
$$

Aquí $\rho$ representa el coeficiente de correlación poblacional de Pearson.

#### Nivel de significancia

Generalmente utilizamos:

$$
\alpha = 0.05
$$

La prueba proporciona un **valor p** (*p-value*).

- **p < 0.05:** rechazamos $H_0$. Existe evidencia estadística de una correlación diferente de cero.
- **p ≥ 0.05:** no rechazamos $H_0$. No tenemos evidencia estadística suficiente para afirmar que la correlación es diferente de cero.

El valor p no indica la fuerza de la correlación ni la probabilidad de que la hipótesis nula sea verdadera.

**Importante:** una correlación estadísticamente significativa no implica que sea fuerte, relevante o causal.

### 3.7. Cálculo de correlaciones con Python

Calcularemos los coeficientes de Pearson y Spearman para la temperatura y la humedad relativa.

Utilizaremos las observaciones horarias de MeteoChillán.

Para cada cálculo:

1. Seleccionaremos las dos variables.
2. Excluiremos únicamente los registros donde falte alguna de ellas.
3. Calcularemos el coeficiente de correlación.
4. Obtendremos el valor p de la prueba estadística.
5. Compararemos los resultados de Pearson y Spearman.

Recordemos que el valor p convencional debe interpretarse con precaución debido a la dependencia temporal de las observaciones meteorológicas.

In [ ]:
# Seleccionar pares válidos
pares = df_final[['Temperatura', 'Humedad']].dropna()

# Pearson
r, p_pearson = pearsonr(
    pares['Temperatura'],
    pares['Humedad']
)

# Spearman
rho, p_spearman = spearmanr(
    pares['Temperatura'],
    pares['Humedad']
)

# Resultados
print(f"Número de pares: {len(pares)}")

print("\nPearson:")
print(f"r = {r:.3f}")
print(f"p = {p_pearson:.3e}")

print("\nSpearman:")
print(f"ρ = {rho:.3f}")
print(f"p = {p_spearman:.3e}")

### 3.7.1. Interpretación de los resultados: temperatura y humedad relativa

Los coeficientes calculados para las observaciones horarias de MeteoChillán fueron:

| Método | Coeficiente | Valor p |
|---|---:|---:|
| Pearson | -0.867 | < 0.001 |
| Spearman | -0.829 | < 0.001 |

**1. Dirección y fuerza de la relación**

Ambos coeficientes son negativos y presentan valores absolutos superiores a 0.80, lo que indica una **asociación negativa muy fuerte**.

- **Pearson (r = -0.867):** indica una relación lineal negativa muy fuerte.
- **Spearman (ρ = -0.829):** indica una asociación monótona negativa muy fuerte.

Esto significa que las temperaturas más altas tienden a coincidir con valores más bajos de humedad relativa.

**2. ¿Por qué Pearson y Spearman presentan valores diferentes?**

Pearson utiliza los valores originales y mide la asociación lineal, mientras que Spearman utiliza los rangos de las observaciones y mide la asociación monótona.

La diferencia entre ambos coeficientes puede deberse a la forma de la relación, la dispersión de los datos y la influencia de valores extremos.

**3. Interpretación física**

La humedad relativa depende del contenido de vapor de agua y de la temperatura.

Cuando aumenta la temperatura, también aumenta la presión de vapor de saturación. Si el contenido de vapor de agua permanece aproximadamente constante, la humedad relativa disminuye.

Esto ayuda a explicar la relación negativa observada, aunque otros procesos atmosféricos también pueden influir en la humedad relativa.

**4. ¿Qué significa el valor p?**

Python devuelve `p = 0.000e+00` porque el valor calculado es extremadamente pequeño y se representa como cero debido a los límites de precisión numérica.

Podemos reportarlo como **p < 0.001**.

Bajo los supuestos de las pruebas convencionales, este resultado indica evidencia estadística contra la hipótesis nula de correlación igual a cero.

Sin embargo, las observaciones meteorológicas horarias presentan **autocorrelación temporal**: los valores registrados en horas consecutivas suelen estar relacionados.

Por tanto, las pruebas convencionales pueden producir valores p demasiado pequeños si consideran todas las observaciones como independientes.

**5. Correlación no implica causalidad**

Aunque encontramos una asociación negativa muy fuerte, esto no demuestra que todos los cambios de humedad relativa sean causados exclusivamente por la temperatura.

También intervienen procesos como el transporte de humedad, la evaporación, la condensación y los cambios de masa de aire.

Además, los ciclos diarios y estacionales pueden contribuir a la asociación observada.


#### Pregunta para discutir

¿Obtendríamos el mismo coeficiente si analizáramos únicamente los meses de verano o invierno?



### 3.8. Matriz de correlación

Hasta ahora calculamos los coeficientes de Pearson y Spearman para temperatura y humedad relativa.

Cuando tenemos varias variables, podemos calcular sus correlaciones mediante una **matriz de correlación**.

Esta matriz presenta los coeficientes de correlación entre todos los pares de variables.

**Características:**

- Los valores se encuentran entre −1 y +1.
- La diagonal principal contiene valores iguales a 1, porque cada variable está perfectamente correlacionada consigo misma.
- La matriz es simétrica: la correlación entre temperatura y humedad es igual a la correlación entre humedad y temperatura.
- Los valores positivos indican asociaciones positivas y los negativos indican asociaciones negativas.

Calcularemos dos matrices:

1. **Pearson:** para evaluar asociaciones lineales.
2. **Spearman:** para evaluar asociaciones monótonas.

Posteriormente, representaremos ambas matrices mediante mapas de calor (*heatmaps*) para facilitar su interpretación.

In [ ]:
# Seleccionar variables meteorológicas
variables = [
    'Temperatura',
    'Humedad',
    'Radiación',
    'Viento',
    'Presión'
]

# Matriz de correlación de Pearson
corr_pearson = df_final[variables].corr(method='pearson')

# Matriz de correlación de Spearman
corr_spearman = df_final[variables].corr(method='spearman')

print("Matriz de correlación de Pearson:")
display(corr_pearson.round(3))

print("\nMatriz de correlación de Spearman:")
display(corr_spearman.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pearson
sns.heatmap(
    corr_pearson,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    cbar_kws={'label': 'Correlación de Pearson (-)'},
    ax=axes[0]
)


# Spearman
sns.heatmap(
    corr_spearman,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    cbar_kws={'label': 'Correlación de Spearman (-)'},
    ax=axes[1]
)


plt.tight_layout()
plt.show()

### 3.8.1. Actividad: interpretación de las matrices de correlación

Observa los mapas de calor de Pearson y Spearman y responde:

1. **Correlación positiva:** ¿Qué par de variables presenta la correlación positiva más fuerte? ¿Qué explicación física podría tener?

2. **Correlación negativa:** ¿Qué par de variables presenta la correlación negativa más fuerte?

3. **Correlación débil:** ¿Qué par presenta el coeficiente más cercano a cero? ¿Significa necesariamente que las variables no están relacionadas?

4. **Comparación de métodos:** ¿Qué pares presentan las mayores diferencias entre Pearson y Spearman? ¿Qué características de los diagramas de dispersión podrían explicar esas diferencias?

5. **Interpretación física:** Selecciona dos relaciones y propone una explicación basada en procesos meteorológicos.

6. **Causalidad:** ¿Podemos determinar cuáles variables causan cambios en otras únicamente a partir de estas matrices?

**Importante:** las matrices permiten identificar asociaciones, pero no demuestran relaciones causales. Además, los ciclos diarios y estacionales pueden influir en los coeficientes calculados.

### 3.9. Correlación no implica causalidad

Hasta ahora hemos utilizado diagramas de dispersión y coeficientes de Pearson y Spearman para identificar relaciones entre variables meteorológicas.

Sin embargo, debemos distinguir dos conceptos:

- **Correlación:** dos variables presentan una asociación estadística; sus valores tienden a variar conjuntamente.
- **Causalidad:** un cambio en una variable produce un cambio en otra, mediante un mecanismo causal.

**Una correlación fuerte, incluso cuando es estadísticamente significativa, no demuestra causalidad.**

Por ejemplo, encontramos una correlación de Pearson de **−0.867** entre temperatura y humedad relativa.

Existe una explicación física para esta relación: al aumentar la temperatura, aumenta la presión de vapor de saturación y, si el contenido de vapor de agua permanece constante, disminuye la humedad relativa.

Sin embargo, esto no significa que todos los cambios de humedad relativa sean causados exclusivamente por la temperatura. También intervienen otros procesos atmosféricos.

Para interpretar correctamente una correlación debemos considerar los mecanismos físicos, las variables adicionales y las características de los datos.

### 3.9.1. Relaciones directas, variables de confusión y correlaciones espurias

Una correlación observada puede tener diferentes explicaciones.

#### A. Relación causal directa

Una variable influye físicamente sobre otra.

**Ejemplo: temperatura y humedad relativa**

Si el contenido de vapor de agua permanece constante, un aumento de temperatura produce una disminución de la humedad relativa.

Temperatura ↑ → Humedad relativa ↓

Existe un mecanismo físico conocido que permite explicar esta relación.

Sin embargo, la correlación observada por sí sola no demuestra cuánto de la variación de humedad relativa se debe a este mecanismo.

#### B. Variable de confusión

Una tercera variable influye sobre las dos variables que estamos comparando.

**Ejemplo: radiación solar y temperatura**

La radiación solar y la temperatura del aire pueden presentar una correlación positiva.

Sin embargo, ambas también dependen de la **hora del día**:

- Durante la noche, la radiación solar es prácticamente cero y las temperaturas suelen ser menores.
- Durante el día, aumenta la radiación solar y generalmente aumenta la temperatura.

La hora del día puede contribuir a la correlación observada.

Además, existe un mecanismo físico real: la energía solar contribuye al calentamiento de la superficie y del aire.

Por tanto, una asociación puede reflejar simultáneamente mecanismos causales y efectos de otras variables.

#### C. Correlación espuria

Dos variables pueden presentar una correlación sin que exista una relación causal directa entre ellas.

**Ejemplo hipotético: ventas de helados y temperatura del aire**

Durante los meses cálidos:

- Aumenta la temperatura.
- Generalmente aumentan las ventas de helados.

Podríamos encontrar una correlación positiva entre ambas variables.

Sin embargo, la temperatura no determina por sí sola las ventas. También intervienen el comportamiento de las personas, las vacaciones y otros factores.

Otro ejemplo sería encontrar una correlación entre las ventas de helados y la cantidad de personas que visitan piscinas. Ambas pueden aumentar durante el verano, sin que necesariamente una cause la otra.

**Importante:** una correlación espuria no significa que el cálculo sea incorrecto. Significa que interpretar esa asociación como una relación causal directa sería injustificado.

### 3.9.2. ¿Cómo podemos investigar una relación causal?

Una correlación puede ser el punto de partida de una investigación, pero no es suficiente para demostrar causalidad.

Para evaluar una posible relación causal debemos considerar:

1. **Mecanismo físico:** ¿existe una explicación científica de cómo una variable puede afectar a la otra?

2. **Orden temporal:** ¿la posible causa ocurre antes que el efecto?

3. **Variables adicionales:** ¿existen otros factores que podrían explicar la asociación observada?

4. **Consistencia:** ¿la relación se mantiene al analizar diferentes periodos o condiciones?

5. **Evidencia adicional:** ¿podemos utilizar experimentos, modelos físicos u otros métodos para evaluar la hipótesis causal?

Por ejemplo, si encontramos una correlación positiva entre radiación solar y temperatura, podríamos investigar:

- ¿La temperatura responde inmediatamente a los cambios de radiación?
- ¿Existe un retraso temporal entre ambas variables?
- ¿La relación cambia entre verano e invierno?
- ¿Qué ocurre durante días nublados?

Estas preguntas permiten avanzar desde una descripción estadística hacia la comprensión de los procesos físicos.



## 4. Conclusiones

En este notebook exploramos las relaciones entre cinco variables meteorológicas registradas por MeteoChillán: temperatura, humedad relativa, radiación solar, velocidad del viento y presión atmosférica.

A partir de datos horarios, utilizamos herramientas de Python para visualizar, cuantificar e interpretar las asociaciones entre estas variables.

### ¿Qué aprendimos?

1. **Exploración visual:** los diagramas de dispersión permiten identificar relaciones positivas, negativas, lineales y no lineales, así como posibles valores extremos.

2. **Correlación de Pearson:** mide la dirección y fuerza de una asociación lineal entre dos variables.

3. **Correlación de Spearman:** mide la dirección y fuerza de una asociación monótona, utilizando los rangos de las observaciones.

4. **Significancia estadística:** las pruebas de hipótesis permiten evaluar la evidencia de una correlación diferente de cero, siempre que se consideren sus supuestos. Un valor p pequeño no significa necesariamente que la relación sea fuerte o importante.

5. **Matrices de correlación:** permiten comparar simultáneamente las asociaciones entre múltiples variables e identificar patrones que merecen una investigación más detallada.

6. **Correlación y causalidad:** una asociación estadística no demuestra que una variable cause cambios en otra. Es necesario considerar los mecanismos físicos, las variables de confusión y otras fuentes de evidencia.


### Reflexión final

**Encontrar una correlación es el comienzo del análisis, no su conclusión.**

El objetivo de la ciencia de datos no es únicamente calcular indicadores estadísticos, sino utilizarlos para formular preguntas, identificar patrones y comprender los procesos que generan los datos.

En ciencias ambientales, una interpretación adecuada requiere combinar las herramientas estadísticas con el conocimiento físico de los fenómenos estudiados.
